## SRP445679

**paper:** [PMID: 39480920](https://pmc.ncbi.nlm.nih.gov/articles/PMC7617403/) - A molecular mechanism for bright color variation in parrots, 2025

**date, curator:** 2026-04-24, Sara Carsanaro

**notes**
* emailed authors about barcode file for scRNA 10X - SRX21829503 and SRX21829504 - april 27th
    * jcorbo@wustl.edu pmaraujo.ecotop@gmail.com roberto.arbore@cibio.up.pt miguel.carneiro@cibio.up.pt 
* rejected lots of libraries
    * WGS libraries
    * ATAC-seq
    * IsoSeq
* annotated 8 bulk libraries for Pseudeos fuscata, although I do not believe that this is a species that bgee excepts
    * SRA says blood but methods clearly state this is feather follicles from the chest - see bulk RNA-seq in methods section and table S8 in supplemental methods

### annotation summary

In [34]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,regenerating feather follicles from chest,UBERON:0011782,feather follicle,perfect match


In [35]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,adult,UBERON:0000113,post-juvenile adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP445679"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [4]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 76/76 [01:17<00:00,  1.02s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [16]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX20771497,SRP445679,Illumina NovaSeq 6000,SRS18057709,,,,,,regenerating feather follicles from chest,adult,,,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
1,SRX20771496,SRP445679,Illumina NovaSeq 6000,SRS18057709,,,,,,regenerating feather follicles from chest,adult,,,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
2,SRX20771503,SRP445679,Illumina NovaSeq 6000,SRS18057711,,,,,,regenerating feather follicles from chest,adult,,,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris25,SAMN35848059,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris25,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX20771504,SRP445679,Illumina NovaSeq 6000,SRS18057713,,,,,,regenerating feather follicles from chest,adult,,,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris26,SAMN35848060,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris26,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX20771499,SRP445679,Illumina NovaSeq 6000,SRS18057715,,,,,,regenerating feather follicles from chest,adult,,,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris28,SAMN35848062,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris28,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX20771500,SRP445679,Illumina NovaSeq 6000,SRS18057717,,,,,,regenerating feather follicles from chest,adult,,,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris30,SAMN35848064,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris30,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
6,SRX20771502,SRP445679,Illumina NovaSeq 6000,SRS18057718,,,,,,regenerating feather follicles from chest,adult,,,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
7,SRX20771501,SRP445679,Illumina NovaSeq 6000,SRS18057718,,,,,,regenerating feather follicles from chest,adult,,,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [17]:
unique_sorted(library, "infoOrgan")

['regenerating feather follicles from chest']


In [18]:

# all
library.loc[:,'anatId'] = 'UBERON:0011782'
library.loc[:,'anatName'] = 'feather follicle'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX20771497,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,,,,regenerating feather follicles from chest,adult,perfect match,not documented,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
1,SRX20771496,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,,,,regenerating feather follicles from chest,adult,perfect match,not documented,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
2,SRX20771503,SRP445679,Illumina NovaSeq 6000,SRS18057711,UBERON:0011782,feather follicle,,,,regenerating feather follicles from chest,adult,perfect match,not documented,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris25,SAMN35848059,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris25,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX20771504,SRP445679,Illumina NovaSeq 6000,SRS18057713,UBERON:0011782,feather follicle,,,,regenerating feather follicles from chest,adult,perfect match,not documented,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris26,SAMN35848060,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris26,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX20771499,SRP445679,Illumina NovaSeq 6000,SRS18057715,UBERON:0011782,feather follicle,,,,regenerating feather follicles from chest,adult,perfect match,not documented,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris28,SAMN35848062,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris28,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX20771500,SRP445679,Illumina NovaSeq 6000,SRS18057717,UBERON:0011782,feather follicle,,,,regenerating feather follicles from chest,adult,perfect match,not documented,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris30,SAMN35848064,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris30,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
6,SRX20771502,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,,,,regenerating feather follicles from chest,adult,perfect match,not documented,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
7,SRX20771501,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,,,,regenerating feather follicles from chest,adult,perfect match,not documented,,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [8]:
unique_sorted(library, "infoStage")

['adult']


In [19]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX20771497,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
1,SRX20771496,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
2,SRX20771503,SRP445679,Illumina NovaSeq 6000,SRS18057711,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris25,SAMN35848059,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris25,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX20771504,SRP445679,Illumina NovaSeq 6000,SRS18057713,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris26,SAMN35848060,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris26,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX20771499,SRP445679,Illumina NovaSeq 6000,SRS18057715,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris28,SAMN35848062,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris28,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX20771500,SRP445679,Illumina NovaSeq 6000,SRS18057717,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris30,SAMN35848064,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris30,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
6,SRX20771502,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
7,SRX20771501,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,unknown,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [20]:
library.loc[:,'sex'] = 'NA'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX20771497,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
1,SRX20771496,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
2,SRX20771503,SRP445679,Illumina NovaSeq 6000,SRS18057711,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris25,SAMN35848059,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris25,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX20771504,SRP445679,Illumina NovaSeq 6000,SRS18057713,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris26,SAMN35848060,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris26,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX20771499,SRP445679,Illumina NovaSeq 6000,SRS18057715,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris28,SAMN35848062,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris28,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX20771500,SRP445679,Illumina NovaSeq 6000,SRS18057717,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris30,SAMN35848064,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris30,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
6,SRX20771502,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
7,SRX20771501,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT


#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [21]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq Stranded mRNA'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX20771497,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
1,SRX20771496,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
2,SRX20771503,SRP445679,Illumina NovaSeq 6000,SRS18057711,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris25,SAMN35848059,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris25,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX20771504,SRP445679,Illumina NovaSeq 6000,SRS18057713,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris26,SAMN35848060,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris26,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX20771499,SRP445679,Illumina NovaSeq 6000,SRS18057715,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris28,SAMN35848062,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris28,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX20771500,SRP445679,Illumina NovaSeq 6000,SRS18057717,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris30,SAMN35848064,,,,,,,,red,,,24/04/2026,TruSeq Stranded mRNA kit,,loris30,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
6,SRX20771502,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
7,SRX20771501,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,,24/04/2026,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT


#### globin, replicates

In [22]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

    #libraryId        SRSId
0  SRX20771497  SRS18057709
1  SRX20771496  SRS18057709
6  SRX20771502  SRS18057718
7  SRX20771501  SRS18057718


/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_10215/3311601719.py:43: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dups = df[duplicateCheck].loc[:,['#libraryId', column]]


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [23]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-27'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX20771497,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,SAC,2026-04-27,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
1,SRX20771496,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,,red,,SAC,2026-04-27,TruSeq Stranded mRNA kit,,loris23,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
2,SRX20771503,SRP445679,Illumina NovaSeq 6000,SRS18057711,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris25,SAMN35848059,,,,,,,,yellow,,SAC,2026-04-27,TruSeq Stranded mRNA kit,,loris25,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX20771504,SRP445679,Illumina NovaSeq 6000,SRS18057713,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris26,SAMN35848060,,,,,,,,yellow,,SAC,2026-04-27,TruSeq Stranded mRNA kit,,loris26,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX20771499,SRP445679,Illumina NovaSeq 6000,SRS18057715,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris28,SAMN35848062,,,,,,,,red,,SAC,2026-04-27,TruSeq Stranded mRNA kit,,loris28,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX20771500,SRP445679,Illumina NovaSeq 6000,SRS18057717,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris30,SAMN35848064,,,,,,,,red,,SAC,2026-04-27,TruSeq Stranded mRNA kit,,loris30,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
6,SRX20771502,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,SAC,2026-04-27,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
7,SRX20771501,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,,yellow,,SAC,2026-04-27,TruSeq Stranded mRNA kit,,loris31,,,,adult,,TRANSCRIPTOMIC,Oligo-dT


#### comments

In [24]:
library.loc[:,'comment'] = 'PMID: 39480920'

#### save complete file with correct columns

In [25]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX20771497,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,PMID: 39480920,red,,SAC,2026-04-27
1,SRX20771496,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,PMID: 39480920,red,,SAC,2026-04-27
2,SRX20771503,SRP445679,Illumina NovaSeq 6000,SRS18057711,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris25,SAMN35848059,,,,,,,PMID: 39480920,yellow,,SAC,2026-04-27
3,SRX20771504,SRP445679,Illumina NovaSeq 6000,SRS18057713,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris26,SAMN35848060,,,,,,,PMID: 39480920,yellow,,SAC,2026-04-27
4,SRX20771499,SRP445679,Illumina NovaSeq 6000,SRS18057715,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris28,SAMN35848062,,,,,,,PMID: 39480920,red,,SAC,2026-04-27
5,SRX20771500,SRP445679,Illumina NovaSeq 6000,SRS18057717,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris30,SAMN35848064,,,,,,,PMID: 39480920,red,,SAC,2026-04-27
6,SRX20771502,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,PMID: 39480920,yellow,,SAC,2026-04-27
7,SRX20771501,SRP445679,Illumina NovaSeq 6000,SRS18057718,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,2,loris31,SAMN35848065,,,,,,,PMID: 39480920,yellow,,SAC,2026-04-27


### experiment annotations

In [26]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP445679,Psittacofulvin coloration in parrots,"In this project, we aim to elucidate the molecular and biochemical mechanisms underlying the rapid diversification of psittacofulvin-based coloration in parrots.",SRA,,,,"NEBNext Single Cell/Low Input RNA Library Prep Kit, TruSeq Stranded mRNA",full_length,,PRJNA986688,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [27]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

8

In [28]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP445679,Psittacofulvin coloration in parrots,"In this project, we aim to elucidate the molecular and biochemical mechanisms underlying the rapid diversification of psittacofulvin-based coloration in parrots.",SRA,partial,Bgee 1K,8,TruSeq Stranded mRNA,full_length,,PRJNA986688,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [29]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '39480920'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7617403/'
experiment.loc[:,'DOI'] = '10.1126/science.adp7710'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP445679,Psittacofulvin coloration in parrots,"In this project, we aim to elucidate the molecular and biochemical mechanisms underlying the rapid diversification of psittacofulvin-based coloration in parrots.",SRA,partial,Bgee 1K,8,TruSeq Stranded mRNA,full_length,,PRJNA986688,39480920,https://pmc.ncbi.nlm.nih.gov/articles/PMC7617403/,10.1126/science.adp7710,,


#### comments

In [30]:
experiment.loc[:,'comment'] = 'for scRNA libraries the authors have been contacted due to missing barcodes. rejected WGS, ATAC, PacBio libraries'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP445679,Psittacofulvin coloration in parrots,"In this project, we aim to elucidate the molecular and biochemical mechanisms underlying the rapid diversification of psittacofulvin-based coloration in parrots.",SRA,partial,Bgee 1K,8,TruSeq Stranded mRNA,full_length,,PRJNA986688,39480920,https://pmc.ncbi.nlm.nih.gov/articles/PMC7617403/,10.1126/science.adp7710,,"for scRNA libraries the authors have been contacted due to missing barcodes. rejected WGS, ATAC, PacBio libraries"


#### save complete file

In [31]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [32]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [33]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [36]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [37]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
62463,SRX2633650,SRP101679,NextSeq 500,SRS2042880,UBERON:0014480,blood feather,UBERON:0000113,post-juvenile adult stage,,regenerating countour feather (non-flight feat...,2,perfect match,not documented,other,F,,,13146,TruSeq Stranded mRNA,full_length,polyA,,,TCR3,SAMN06555321,2,year,,,,,"PMID: 28985565, i think it makes sense to anno...",yellow feather pigmentation phenotype (wild type),,SAC,2026-04-24
62464,SRX2633649,SRP101679,NextSeq 500,SRS2042879,UBERON:0014480,blood feather,UBERON:0000113,post-juvenile adult stage,,regenerating countour feather (non-flight feat...,1,perfect match,not documented,other,M,,,13146,TruSeq Stranded mRNA,full_length,polyA,,,TCR9,SAMN06555323,1,year,,,,,"PMID: 28985565, i think it makes sense to anno...",blue feather pigmentation phenotype (MuPKS R644W),,SAC,2026-04-24
62465,SRX20771497,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,PMID: 39480920,red,,SAC,2026-04-27
62466,SRX20771496,SRP445679,Illumina NovaSeq 6000,SRS18057709,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,1,loris23,SAMN35848057,,,,,,,PMID: 39480920,red,,SAC,2026-04-27
62467,SRX20771503,SRP445679,Illumina NovaSeq 6000,SRS18057711,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris25,SAMN35848059,,,,,,,PMID: 39480920,yellow,,SAC,2026-04-27
62468,SRX20771504,SRP445679,Illumina NovaSeq 6000,SRS18057713,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris26,SAMN35848060,,,,,,,PMID: 39480920,yellow,,SAC,2026-04-27
62469,SRX20771499,SRP445679,Illumina NovaSeq 6000,SRS18057715,UBERON:0011782,feather follicle,UBERON:0000113,post-juvenile adult stage,,regenerating feather follicles from chest,adult,perfect match,not documented,perfect match,NA,,,176072,TruSeq Stranded mRNA,full_length,polyA,,,loris28,SAMN35848062,,,,,,,PMID: 39480920,red,,SAC,2026-04-27


In [38]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1206,SRP663430,IAP retrotransposons contribute to the transcr...,Transposable elements (TEs) have made importan...,SRA,partial,Bgee 1K,9,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Di...",full_length,,PRJNA1403818,,https://www.biorxiv.org/content/10.64898/2026....,10.64898/2026.01.27.702056,,"removed cell culture libraries. no PMID yet, s..."
1207,SRP101679,Melopsittacus undulatus pigmentation trait map...,Budgerigars exhibit several Mendelian feather ...,SRA,partial,Bgee 1K,7,TruSeq Stranded mRNA,full_length,,PRJNA378643,28985565,https://pmc.ncbi.nlm.nih.gov/articles/PMC5951300/,10.1016/j.cell.2017.08.016,,rejected all DNA seq libraries
1208,SRP445679,Psittacofulvin coloration in parrots,"In this project, we aim to elucidate the molec...",SRA,partial,Bgee 1K,8,TruSeq Stranded mRNA,full_length,,PRJNA986688,39480920,https://pmc.ncbi.nlm.nih.gov/articles/PMC7617403/,10.1126/science.adp7710,,for scRNA libraries the authors have been cont...


### add annotations to git

In [39]:
! git pull

Already up to date.


In [40]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [41]:
! git add $git_experiment_path $git_library_path

In [42]:
! git commit -m $commit_message_exp

[develop d0d848a] adding annotated bulk experiment SRP445679
 2 files changed, 9 insertions(+)


In [43]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.08 KiB | 2.08 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   6eb2da3..d0d848a  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push